In [ ]:
#@title Gemini Live Transcriber { display-mode: "form" }
# One run. Prompts: model -> API key -> upload -> automatic transcription.

import os, sys, re, math, time, json, shutil, asyncio, subprocess, getpass, zipfile
from array import array
from pathlib import Path
from datetime import datetime
from google.colab import files

SAMPLE_RATE = 16_000
BYTES_PER_SAMPLE = 2
BYTES_PER_SEC = SAMPLE_RATE * BYTES_PER_SAMPLE
FRAME_MS = 100
FRAME_BYTES = BYTES_PER_SEC * FRAME_MS // 1000
OVERLAP_SEC = 1.0
STALL_ACTIVE_AUDIO_SEC = 12.0
STALL_RMS_THRESHOLD = 160
FINAL_IDLE_SEC = 6.0
TMP = Path('/content/gemini_live_tmp')
TMP.mkdir(parents=True, exist_ok=True)

MODELS = {
    '1': {
        'name': 'Gemini 3.5 Transcribe Live',
        'id': 'gemini-3.5-transcribe-live',
        'tpm': 20_000,
        'concurrency': 2,
        'max_chunk_sec': 540,
    },
    '2': {
        'name': 'Gemini 3.8 Live',
        'id': 'gemini-3.8-live',
        'tpm': 65_000,
        'concurrency': 40,
        'max_chunk_sec': 480,
    },
}


def choose_models():
    print('Choose model:')
    print('  1 = Gemini 3.5 Transcribe Live')
    print('  2 = Gemini 3.8 Live')
    print('  3 = Both (3.5 first, then 3.8)')
    while True:
        choice = input('Model [1]: ').strip() or '1'
        if choice in ('1', '2', '3'):
            return ['1', '2'] if choice == '3' else [choice]
        print('Type 1, 2, or 3.')


def ask_key():
    while True:
        key = getpass.getpass('Gemini API key: ').strip()
        if key:
            return key
        print('API key cannot be empty.')


def upload_one_file():
    print('\nChoose the recording to upload...')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    if len(uploaded) != 1:
        raise RuntimeError('Please upload exactly one recording.')
    name, data = next(iter(uploaded.items()))
    path = Path('/content') / Path(name).name
    path.write_bytes(data)
    return path


def install_sdk():
    print('\n[setup] Preparing Gemini SDK...', flush=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'google-genai==2.25.0'],
        check=True,
    )
    global genai, types
    from google import genai as _genai
    from google.genai import types as _types
    genai, types = _genai, _types


def ffmpeg_to_pcm(src, dst):
    if not shutil.which('ffmpeg'):
        raise RuntimeError('ffmpeg is unavailable in this Colab runtime.')
    subprocess.run([
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-i', str(src), '-vn', '-ac', '1', '-ar', str(SAMPLE_RATE),
        '-f', 's16le', str(dst),
    ], check=True)
    if not dst.exists() or dst.stat().st_size == 0:
        raise RuntimeError('Audio conversion failed or produced an empty file.')
    return dst.stat().st_size / BYTES_PER_SEC


def hms(sec):
    sec = max(0, int(round(sec)))
    h, rem = divmod(sec, 3600)
    m, s = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'


def safe_name(s):
    return re.sub(r'[^\w.-]+', '_', s, flags=re.UNICODE).strip('_') or 'transcript'


def make_chunks(total_sec, spec):
    c = min(spec['concurrency'], max(1, math.ceil(total_sec / 2)))
    waves = max(1, math.ceil(total_sec / (c * spec['max_chunk_sec'])))
    count = min(max(1, waves * c), max(1, math.ceil(total_sec / 2)))
    chunk_len = total_sec / count
    out = []
    for i in range(count):
        nominal_start = i * chunk_len
        nominal_end = total_sec if i == count - 1 else (i + 1) * chunk_len
        out.append({
            'index': i,
            'start_sec': max(0.0, nominal_start - (OVERLAP_SEC if i else 0.0)),
            'end_sec': min(total_sec, nominal_end + (OVERLAP_SEC if i < count - 1 else 0.0)),
        })
    return out


def norm_word(w):
    return re.sub(r'[^\w\u0600-\u06FF]+', '', w, flags=re.UNICODE).casefold()


def merge_overlap(parts, max_words=60):
    merged = []
    for text in parts:
        words = text.strip().split()
        if not words:
            continue
        if not merged:
            merged = words
            continue
        left = [norm_word(x) for x in merged]
        right = [norm_word(x) for x in words]
        best = 0
        for n in range(min(max_words, len(merged), len(words)), 1, -1):
            if left[-n:] == right[:n] and any(left[-n:]):
                best = n
                break
        merged.extend(words[best:])
    return ' '.join(merged).strip()


def pcm_rms(data):
    if len(data) < 2:
        return 0.0
    samples = array('h')
    samples.frombytes(data[:len(data) - (len(data) % 2)])
    if sys.byteorder != 'little':
        samples.byteswap()
    if not samples:
        return 0.0
    return math.sqrt(sum(v * v for v in samples) / len(samples))


def probably_silent(path, start_byte, end_byte, threshold=120):
    span = max(0, end_byte - start_byte)
    if span <= 0:
        return True
    window = min(BYTES_PER_SEC // 2, span)
    positions = [start_byte] if span <= window else [
        start_byte + int((span - window) * i / 7) for i in range(8)
    ]
    peak = 0.0
    with open(path, 'rb') as f:
        for pos in positions:
            pos -= pos % 2
            f.seek(pos)
            data = f.read(window)
            if len(data) < 2:
                continue
            samples = array('h')
            samples.frombytes(data[:len(data) - (len(data) % 2)])
            if sys.byteorder != 'little':
                samples.byteswap()
            if samples:
                rms = math.sqrt(sum(v * v for v in samples) / len(samples))
                peak = max(peak, rms)
    return peak < threshold


def model_config(key):
    if key == '1':
        return {
            'response_modalities': ['TEXT'],
            'realtime_input_config': {
                'automatic_activity_detection': {'disabled': True},
            },
            'input_audio_transcription': {
                'language_codes': [],
                'mode': 'VERBATIM',
            },
        }
    return {
        'response_modalities': ['AUDIO'],
        'input_audio_transcription': {},
    }


async def receive_transcript(session, stream_done, telemetry):
    pieces = []
    iterator = session.receive().__aiter__()
    while True:
        timeout = 30.0 if not stream_done.is_set() else (FINAL_IDLE_SEC if pieces else 20.0)
        try:
            msg = await asyncio.wait_for(iterator.__anext__(), timeout=timeout)
        except StopAsyncIteration:
            telemetry['receive_end'] += 1
            break
        except asyncio.TimeoutError:
            telemetry['receive_timeout'] += 1
            break
        except Exception as e:
            code = getattr(e, 'code', None)
            if code == 1000 or str(e).lstrip().startswith('1000'):
                telemetry['normal_close'] += 1
                break
            raise

        telemetry['messages'] += 1
        telemetry['last_server_at'] = time.monotonic()
        sc = getattr(msg, 'server_content', None)
        if not sc:
            telemetry['other_messages'] += 1
            continue

        interim_tr = getattr(sc, 'interim_input_transcription', None)
        if interim_tr and (getattr(interim_tr, 'text', '') or '').strip():
            telemetry['interim_events'] += 1

        tr = getattr(sc, 'input_transcription', None)
        text = (getattr(tr, 'text', '') or '').strip() if tr else ''
        if text:
            telemetry['final_events'] += 1
            if not pieces or text != pieces[-1]:
                pieces.append(text)

    return ' '.join(pieces).strip()


async def transcribe_chunk(client, pcm_path, chunk, model_key, max_attempts=3):
    spec = MODELS[model_key]
    start_byte = int(chunk['start_sec'] * BYTES_PER_SEC)
    end_byte = int(chunk['end_sec'] * BYTES_PER_SEC)
    start_byte -= start_byte % 2
    end_byte -= end_byte % 2
    total = max(0, end_byte - start_byte)
    last_error = None

    for attempt in range(1, max_attempts + 1):
        recv = None
        done = asyncio.Event()
        telemetry = {
            'messages': 0,
            'last_server_at': None,
            'interim_events': 0,
            'final_events': 0,
            'receive_timeout': 0,
            'receive_end': 0,
            'normal_close': 0,
            'other_messages': 0,
        }
        active_without_server = 0.0
        seen_messages = 0
        try:
            async with client.aio.live.connect(model=spec['id'], config=model_config(model_key)) as session:
                recv = asyncio.create_task(receive_transcript(session, done, telemetry))
                if model_key == '1':
                    await session.send_realtime_input(activity_start=types.ActivityStart())
                remaining = total
                with open(pcm_path, 'rb', buffering=1024 * 1024) as fh:
                    fh.seek(start_byte)
                    next_send = time.monotonic()
                    while remaining > 0:
                        data = fh.read(min(FRAME_BYTES, remaining))
                        if not data:
                            break

                        await session.send_realtime_input(
                            audio=types.Blob(data=data, mime_type='audio/pcm;rate=16000')
                        )
                        remaining -= len(data)

                        if telemetry['messages'] != seen_messages:
                            seen_messages = telemetry['messages']
                            active_without_server = 0.0
                        elif model_key == '1' and pcm_rms(data) >= STALL_RMS_THRESHOLD:
                            active_without_server += len(data) / BYTES_PER_SEC
                            if active_without_server >= STALL_ACTIVE_AUDIO_SEC:
                                raise RuntimeError(
                                    'silent Live session: no server responses during '
                                    f'{active_without_server:.1f}s of active audio'
                                )

                        next_send += FRAME_MS / 1000.0
                        delay = next_send - time.monotonic()
                        if delay > 0:
                            await asyncio.sleep(delay)

                if model_key == '1':
                    await session.send_realtime_input(activity_end=types.ActivityEnd())
                else:
                    await session.send_realtime_input(audio_stream_end=True)
                done.set()
                try:
                    text = await asyncio.wait_for(recv, timeout=30.0)
                finally:
                    if recv is not None and not recv.done():
                        recv.cancel()

                if not text and not await asyncio.to_thread(
                    probably_silent, pcm_path, start_byte, end_byte
                ):
                    raise RuntimeError(
                        'empty transcript on non-silent audio; '
                        f'server_messages={telemetry["messages"]}, '
                        f'interim={telemetry["interim_events"]}, '
                        f'final={telemetry["final_events"]}'
                    )
                return text
        except Exception as e:
            last_error = e
            done.set()
            if recv is not None and not recv.done():
                recv.cancel()
                try:
                    await recv
                except BaseException:
                    pass
            if attempt < max_attempts:
                await asyncio.sleep(2 * attempt)

    raise RuntimeError(f'chunk {chunk["index"] + 1} failed: {last_error}')


async def transcribe_model(api_key, pcm_path, duration, model_key):
    spec = MODELS[model_key]
    chunks = make_chunks(duration, spec)
    concurrency = min(spec['concurrency'], len(chunks))
    results = [None] * len(chunks)
    client = genai.Client(api_key=api_key)
    started = time.monotonic()
    done_count = 0
    stop_heartbeat = asyncio.Event()

    expected = max(ch['end_sec'] - ch['start_sec'] for ch in chunks) * math.ceil(len(chunks) / concurrency)
    print(
        f'\n[{spec["name"]}] {len(chunks)} chunks | up to {concurrency} parallel | '
        f'audio {hms(duration)} | rough minimum ~{expected/60:.1f} min',
        flush=True,
    )

    async def heartbeat():
        while not stop_heartbeat.is_set():
            elapsed = time.monotonic() - started
            print(
                f'\r[{spec["name"]}] working... {done_count}/{len(chunks)} chunks | elapsed {hms(elapsed)}',
                end='', flush=True,
            )
            try:
                await asyncio.wait_for(stop_heartbeat.wait(), timeout=10)
            except asyncio.TimeoutError:
                pass

    hb = asyncio.create_task(heartbeat())

    async def worker(ch):
        nonlocal done_count
        text = await transcribe_chunk(client, pcm_path, ch, model_key)
        results[ch['index']] = text
        done_count += 1

    async def run_wave(wave):
        tasks = [asyncio.create_task(worker(ch)) for ch in wave]
        settled = await asyncio.gather(*tasks, return_exceptions=True)
        return [x for x in settled if isinstance(x, Exception)]

    try:
        failures = []
        for offset in range(0, len(chunks), concurrency):
            failures.extend(await run_wave(chunks[offset:offset + concurrency]))

        missing = [ch for ch in chunks if results[ch['index']] is None]
        recovery = max(1, concurrency // 2)
        round_no = 0
        while missing and round_no < 5:
            round_no += 1
            print(f'\n[{spec["name"]}] retrying {len(missing)} failed chunk(s) with concurrency {recovery}...', flush=True)
            for offset in range(0, len(missing), recovery):
                await run_wave(missing[offset:offset + recovery])
            missing = [ch for ch in chunks if results[ch['index']] is None]
            recovery = max(1, recovery // 2)

        if missing:
            raise RuntimeError(
                f'{len(missing)} chunk(s) still failed after retries; no incomplete transcript was saved.'
            )

        transcript = merge_overlap(results)
        elapsed = time.monotonic() - started
        print(f'\r[{spec["name"]}] done: {len(chunks)}/{len(chunks)} chunks | {elapsed/60:.2f} min' + ' ' * 20, flush=True)
        return transcript, elapsed
    finally:
        stop_heartbeat.set()
        try:
            await hb
        except Exception:
            pass
        try:
            client.close()
        except Exception:
            pass


async def main():
    chosen = choose_models()
    api_key = ask_key()
    src = upload_one_file()
    install_sdk()

    pcm = TMP / f'{int(time.time())}_{safe_name(src.stem)}.pcm'
    print('\n[1/3] Preparing audio...', flush=True)
    prep_start = time.monotonic()
    duration = await asyncio.to_thread(ffmpeg_to_pcm, src, pcm)
    print(f'[1/3] Ready: {src.name} | duration {hms(duration)} | prep {time.monotonic()-prep_start:.1f}s', flush=True)

    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    base = safe_name(src.stem)
    outputs = []
    timings = []

    try:
        print('[2/3] Transcribing...', flush=True)
        for key in chosen:
            text, elapsed = await transcribe_model(api_key, pcm, duration, key)
            spec = MODELS[key]
            out = Path('/content') / f'{base}_{spec["id"]}_{stamp}.txt'
            out.write_text(text, encoding='utf-8')
            outputs.append(out)
            timings.append((spec['name'], elapsed))

        print('\n[3/3] Finished.', flush=True)
        for name, elapsed in timings:
            print(f'  {name}: {elapsed/60:.2f} min')

        if len(outputs) == 1:
            print(f'\nDownloading: {outputs[0].name}', flush=True)
            files.download(str(outputs[0]))
        else:
            z = Path('/content') / f'{base}_Gemini_Live_transcripts_{stamp}.zip'
            with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as archive:
                for p in outputs:
                    archive.write(p, arcname=p.name)
            print(f'\nDownloading: {z.name}', flush=True)
            files.download(str(z))
    finally:
        try:
            pcm.unlink(missing_ok=True)
        except Exception:
            pass


await main()


In [ ]:
#@title Gemini 3.5 Transcribe Live - Diagnostic + Benchmark { display-mode: "form" }
# One run: official Google control -> source diagnosis -> raw WebSocket isolation -> concurrency benchmark.

import sys, math, time, json, base64, asyncio, subprocess, getpass, shutil, urllib.request, urllib.parse, wave, re
from array import array
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from google.colab import files
from IPython.display import Audio, display

MODEL_ID = 'gemini-3.5-transcribe-live'
MODEL_NAME = 'Gemini 3.5 Transcribe Live'
SDK_VERSION = '2.25.0'
SAMPLE_RATE = 16_000
BYTES_PER_SEC = SAMPLE_RATE * 2
FRAME_MS = 100
FRAME_BYTES = BYTES_PER_SEC * FRAME_MS // 1000
PROBE_SEC = 12
CONCURRENCY_PROBE_SEC = 10
SUSTAINED_SEC = 45
INTEGRITY_SEC = 40
TURN_SEC = 10
TURN_FINAL_TIMEOUT_SEC = 10
PROFILES = (2, 4, 6, 8, 10, 12)
COOLDOWN_SEC = 50
FINAL_WAIT_SEC = 30
FINAL_IDLE_SEC = 5
OFFICIAL_SAMPLE_URL = 'https://storage.googleapis.com/generativeai-downloads/audio/tell-a-story.wav'
RAW_WS_URL = 'wss://generativelanguage.googleapis.com/ws/google.ai.generativelanguage.v1beta.GenerativeService.BidiGenerateContent'
TMP = Path('/content/gemini35_live_debug')
TMP.mkdir(parents=True, exist_ok=True)


def ask_key():
    while True:
        key = getpass.getpass('Gemini API key: ').strip()
        if key:
            return key
        print('API key cannot be empty.')


def upload_one_file():
    print('\nChoose the recording to upload...')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    if len(uploaded) != 1:
        raise RuntimeError('Please upload exactly one recording.')
    name, data = next(iter(uploaded.items()))
    path = Path('/content') / Path(name).name
    path.write_bytes(data)
    return path


def install_sdk():
    print('\n[setup] Preparing Google Gen AI SDK...', flush=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', f'google-genai=={SDK_VERSION}'],
        check=True,
    )
    global genai, types, websockets, google_genai_version
    from google import genai as _genai
    from google.genai import types as _types
    import websockets as _websockets
    import importlib.metadata as _metadata
    google_genai_version = _metadata.version('google-genai')
    genai, types, websockets = _genai, _types, _websockets


def ffmpeg_to_pcm(src, dst):
    if not shutil.which('ffmpeg'):
        raise RuntimeError('ffmpeg is unavailable in this Colab runtime.')
    subprocess.run([
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-i', str(src), '-vn', '-ac', '1', '-ar', str(SAMPLE_RATE),
        '-f', 's16le', str(dst),
    ], check=True)
    if not dst.exists() or dst.stat().st_size == 0:
        raise RuntimeError('Audio conversion failed or produced an empty file.')
    return dst.stat().st_size / BYTES_PER_SEC


def audio_stats(path, start_sec, seconds):
    start = int(start_sec * BYTES_PER_SEC) // 2 * 2
    size = int(seconds * BYTES_PER_SEC) // 2 * 2
    with open(path, 'rb') as fh:
        fh.seek(start)
        data = fh.read(size)
    samples = array('h')
    samples.frombytes(data[:len(data) - (len(data) % 2)])
    if sys.byteorder != 'little':
        samples.byteswap()
    if not samples:
        return {'rms': 0.0, 'dbfs': -120.0, 'peak': 0, 'zero_fraction': 1.0}
    rms = math.sqrt(sum(v * v for v in samples) / len(samples))
    peak = max(abs(v) for v in samples)
    dbfs = 20 * math.log10(max(rms, 1e-9) / 32768.0)
    zero_fraction = sum(abs(v) < 8 for v in samples) / len(samples)
    return {'rms': rms, 'dbfs': dbfs, 'peak': peak, 'zero_fraction': zero_fraction}


def candidate_probes(path, duration, seconds, count=12):
    if duration <= seconds:
        return [(0.0, audio_stats(path, 0.0, duration))]
    max_start = duration - seconds
    starts = [max_start * i / (count - 1) for i in range(count)]
    scored = [(s, audio_stats(path, s, seconds)) for s in starts]
    scored.sort(key=lambda item: item[1]['rms'], reverse=True)
    return scored


def write_probe_wav(pcm_path, start_sec, seconds, out_path):
    start = int(start_sec * BYTES_PER_SEC) // 2 * 2
    size = int(seconds * BYTES_PER_SEC) // 2 * 2
    with open(pcm_path, 'rb') as fh:
        fh.seek(start)
        data = fh.read(size)
    with wave.open(str(out_path), 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(SAMPLE_RATE)
        wf.writeframes(data)
    return out_path


def sdk_config(language_codes, manual_vad=False):
    kwargs = {
        'response_modalities': ['TEXT'],
        'input_audio_transcription': types.AudioTranscriptionConfig(language_codes=language_codes),
    }
    if manual_vad:
        kwargs['realtime_input_config'] = types.RealtimeInputConfig(
            automatic_activity_detection=types.AutomaticActivityDetection(disabled=True)
        )
    return types.LiveConnectConfig(**kwargs)


def telemetry(label, start_sec, seconds, transport, vad_mode):
    return {
        'label': label, 'transport': transport, 'vad_mode': vad_mode,
        'start_sec': round(start_sec, 3), 'audio_sec': round(seconds, 3),
        'ok': False, 'error': '', 'messages': 0, 'server_content_messages': 0,
        'interim_events': 0, 'final_events': 0, 'turn_complete': 0,
        'first_message_ms': None, 'total_elapsed_sec': None,
        'final_text': '', 'interim_text': '', 'final_segments': [], 'normal_close': False,
        'receiver_error': '', 'event_types': {},
    }


def append_unique(parts, text):
    text = (text or '').strip()
    if text and (not parts or text != parts[-1]):
        parts.append(text)


async def sdk_receiver(session, t, started):
    finals = []
    try:
        async for msg in session.receive():
            t['messages'] += 1
            if t['first_message_ms'] is None:
                t['first_message_ms'] = round((time.monotonic() - started) * 1000)
            sc = getattr(msg, 'server_content', None)
            if not sc:
                continue
            t['server_content_messages'] += 1
            interim = getattr(sc, 'interim_input_transcription', None)
            final = getattr(sc, 'input_transcription', None)
            if interim and getattr(interim, 'text', None):
                t['interim_events'] += 1
                # Interim is a mutable hypothesis; keep only the latest one.
                t['interim_text'] = interim.text.strip()
            if final and getattr(final, 'text', None):
                text = final.text.strip()
                t['final_events'] += 1
                t['final_segments'].append(text)
                append_unique(finals, text)
                t['final_text'] = ' '.join(finals).strip()
            if getattr(sc, 'turn_complete', False):
                t['turn_complete'] += 1
    except asyncio.CancelledError:
        raise
    except Exception as e:
        code = getattr(e, 'code', None)
        if code == 1000 or str(e).lstrip().startswith('1000'):
            t['normal_close'] = True
        else:
            t['receiver_error'] = f'{type(e).__name__}: {e}'


async def send_sdk_audio(session, pcm_path, start_sec, seconds, manual_vad):
    start = int(start_sec * BYTES_PER_SEC) // 2 * 2
    remaining = int(seconds * BYTES_PER_SEC) // 2 * 2
    if manual_vad:
        await session.send_realtime_input(activity_start=types.ActivityStart())
    with open(pcm_path, 'rb', buffering=1024 * 1024) as fh:
        fh.seek(start)
        next_send = time.monotonic()
        while remaining > 0:
            data = fh.read(min(FRAME_BYTES, remaining))
            if not data:
                break
            await session.send_realtime_input(
                audio=types.Blob(data=data, mime_type='audio/pcm;rate=16000')
            )
            remaining -= len(data)
            next_send += FRAME_MS / 1000.0
            delay = next_send - time.monotonic()
            if delay > 0:
                await asyncio.sleep(delay)
    if manual_vad:
        await session.send_realtime_input(activity_end=types.ActivityEnd())
    else:
        await session.send_realtime_input(audio_stream_end=True)


async def sdk_probe(client, pcm_path, start_sec, seconds, language_codes, manual_vad=False, label='source'):
    t = telemetry(label, start_sec, seconds, 'google-genai', 'manual' if manual_vad else 'hybrid')
    started = time.monotonic()
    recv = None
    try:
        async with client.aio.live.connect(model=MODEL_ID, config=sdk_config(language_codes, manual_vad)) as session:
            recv = asyncio.create_task(sdk_receiver(session, t, started))
            await send_sdk_audio(session, pcm_path, start_sec, seconds, manual_vad)

            # Do not stop on the first final event. Google documents finals as
            # committed speech segments; long turns can emit more than one.
            last_count = t['messages']
            last_change = time.monotonic()
            deadline = time.monotonic() + FINAL_WAIT_SEC
            while time.monotonic() < deadline:
                if recv.done():
                    break
                if t['messages'] != last_count:
                    last_count = t['messages']
                    last_change = time.monotonic()
                if t['final_events'] and time.monotonic() - last_change >= FINAL_IDLE_SEC:
                    break
                await asyncio.sleep(0.25)

            if t['final_text']:
                t['ok'] = True
            elif t['interim_text']:
                t['error'] = 'interim transcription arrived, but no finalized transcript'
            elif t['messages'] == 0:
                t['error'] = 'silent session: zero server messages'
            else:
                t['error'] = 'server messages arrived, but no transcription fields were emitted'

            if t['receiver_error'] and not t['error']:
                t['error'] = t['receiver_error']
                if not t['final_text']:
                    t['ok'] = False
    except Exception as e:
        t['error'] = f'{type(e).__name__}: {e}'
    finally:
        if recv is not None and not recv.done():
            recv.cancel()
            try:
                await recv
            except BaseException:
                pass
        t['total_elapsed_sec'] = round(time.monotonic() - started, 3)
    return t


async def raw_receiver(ws, t):
    finals, interims = [], []
    try:
        async for raw in ws:
            t['messages'] += 1
            msg = json.loads(raw)
            for key in msg:
                t['event_types'][key] = t['event_types'].get(key, 0) + 1
            sc = msg.get('serverContent')
            if not sc:
                continue
            t['server_content_messages'] += 1
            interim = sc.get('interimInputTranscription') or {}
            final = sc.get('inputTranscription') or {}
            if interim.get('text'):
                t['interim_events'] += 1
                t['interim_text'] = interim['text'].strip()
            if final.get('text'):
                t['final_events'] += 1
                append_unique(finals, final['text'])
                t['final_text'] = ' '.join(finals).strip()
            if sc.get('turnComplete'):
                t['turn_complete'] += 1
    except asyncio.CancelledError:
        raise
    except Exception as e:
        code = getattr(e, 'code', None)
        if code == 1000 or str(e).lstrip().startswith('1000'):
            t['normal_close'] = True
        else:
            t['receiver_error'] = f'raw receiver {type(e).__name__}: {e}'


async def raw_ws_probe(api_key, pcm_path, start_sec, seconds, language_codes, label):
    t = telemetry(label, start_sec, seconds, 'raw-websocket', 'hybrid')
    started = time.monotonic()
    recv = None
    uri = RAW_WS_URL + '?key=' + urllib.parse.quote(api_key, safe='')
    try:
        async with websockets.connect(uri, max_size=None, open_timeout=15, close_timeout=5) as ws:
            await ws.send(json.dumps({
                'setup': {
                    'model': f'models/{MODEL_ID}',
                    'generationConfig': {'responseModalities': ['TEXT']},
                    'inputAudioTranscription': {'languageCodes': language_codes},
                }
            }))
            setup_msg = json.loads(await asyncio.wait_for(ws.recv(), timeout=10))
            t['messages'] += 1
            t['first_message_ms'] = round((time.monotonic() - started) * 1000)
            for key in setup_msg:
                t['event_types'][key] = t['event_types'].get(key, 0) + 1
            if 'setupComplete' not in setup_msg:
                raise RuntimeError(f'expected setupComplete, got keys={list(setup_msg)}')
            recv = asyncio.create_task(raw_receiver(ws, t))
            start_byte = int(start_sec * BYTES_PER_SEC) // 2 * 2
            remaining = int(seconds * BYTES_PER_SEC) // 2 * 2
            with open(pcm_path, 'rb', buffering=1024 * 1024) as fh:
                fh.seek(start_byte)
                next_send = time.monotonic()
                while remaining > 0:
                    data = fh.read(min(FRAME_BYTES, remaining))
                    if not data:
                        break
                    await ws.send(json.dumps({
                        'realtimeInput': {
                            'audio': {
                                'data': base64.b64encode(data).decode('ascii'),
                                'mimeType': 'audio/pcm;rate=16000',
                            }
                        }
                    }))
                    remaining -= len(data)
                    next_send += FRAME_MS / 1000.0
                    delay = next_send - time.monotonic()
                    if delay > 0:
                        await asyncio.sleep(delay)
            await ws.send(json.dumps({'realtimeInput': {'audioStreamEnd': True}}))
            deadline = time.monotonic() + FINAL_WAIT_SEC
            while time.monotonic() < deadline:
                if t['final_events']:
                    await asyncio.sleep(2)
                    break
                await asyncio.sleep(0.25)
            if t['final_text']:
                t['ok'] = True
            elif t['interim_text']:
                t['error'] = 'raw WebSocket got interim but no final transcript'
            elif t['messages'] <= 1:
                t['error'] = 'raw WebSocket got setupComplete but no later server messages'
            else:
                t['error'] = 'raw WebSocket got server messages but no transcription fields'
    except Exception as e:
        t['error'] = f'{type(e).__name__}: {e}'
    finally:
        if recv is not None and not recv.done():
            recv.cancel()
            try:
                await recv
            except BaseException:
                pass
        t['total_elapsed_sec'] = round(time.monotonic() - started, 3)
    return t


def print_probe(r):
    print(
        f"  {'PASS' if r['ok'] else 'FAIL'} | {r['label']} | {r['transport']} | VAD={r['vad_mode']} | "
        f"offset={r['start_sec']:.1f}s | total={r['total_elapsed_sec']:.1f}s"
    )
    print(
        f"    messages={r['messages']} server_content={r['server_content_messages']} "
        f"interim={r['interim_events']} final={r['final_events']} first_message_ms={r['first_message_ms']} "
        f"final_chars={len(r['final_text'])} interim_chars={len(r['interim_text'])}"
    )
    if r['final_text']:
        print('    final sample:', r['final_text'][:220].replace('\n', ' '))
    elif r['interim_text']:
        print('    interim sample:', r['interim_text'][:220].replace('\n', ' '))
    if r['event_types']:
        print('    raw event types:', r['event_types'])
    if r['error']:
        print('    error:', r['error'][:500])


def chunk_floor_minutes(duration, concurrency, max_chunk_sec=540):
    waves = max(1, math.ceil(duration / (concurrency * max_chunk_sec)))
    count = waves * concurrency
    chunk_len = duration / count
    return chunk_len * waves / 60.0, count, chunk_len



def normalize_for_compare(text):
    text = re.sub(r'[^\w\u0600-\u06FF]+', ' ', text.casefold(), flags=re.UNICODE)
    return ' '.join(text.split())


def similarity_metrics(reference, candidate):
    import difflib
    ref = normalize_for_compare(reference)
    cand = normalize_for_compare(candidate)
    char_ratio = len(cand) / max(1, len(ref))
    seq_ratio = difflib.SequenceMatcher(None, ref, cand).ratio()
    return {
        'reference_chars': len(ref),
        'candidate_chars': len(cand),
        'length_ratio': round(char_ratio, 3),
        'sequence_ratio': round(seq_ratio, 3),
    }


async def build_independent_reference(client, pcm_path, start_sec, total_sec):
    segments = []
    results = []
    count = math.ceil(total_sec / TURN_SEC)
    for i in range(count):
        seg_start = start_sec + i * TURN_SEC
        seg_sec = min(TURN_SEC, start_sec + total_sec - seg_start)
        if seg_sec <= 0:
            break
        r = await sdk_probe(
            client, pcm_path, seg_start, seg_sec, [], True,
            f'reference turn {i+1}/{count}'
        )
        results.append(r)
        print_probe(r)
        if not r['ok']:
            return {'ok': False, 'text': '', 'results': results}
        segments.append(r['final_text'])
        await asyncio.sleep(1)
    return {'ok': True, 'text': ' '.join(segments).strip(), 'results': results}


async def multi_turn_probe(client, pcm_path, start_sec, total_sec, label):
    t = telemetry(label, start_sec, total_sec, 'google-genai', 'manual-multi-turn')
    t['turns_sent'] = 0
    t['turns_finalized'] = 0
    started = time.monotonic()
    recv = None
    try:
        async with client.aio.live.connect(
            model=MODEL_ID, config=sdk_config([], True)
        ) as session:
            recv = asyncio.create_task(sdk_receiver(session, t, started))
            count = math.ceil(total_sec / TURN_SEC)

            for i in range(count):
                seg_start = start_sec + i * TURN_SEC
                seg_sec = min(TURN_SEC, start_sec + total_sec - seg_start)
                if seg_sec <= 0:
                    break

                before = t['final_events']
                await send_sdk_audio(session, pcm_path, seg_start, seg_sec, True)
                t['turns_sent'] += 1

                deadline = time.monotonic() + TURN_FINAL_TIMEOUT_SEC
                while time.monotonic() < deadline and t['final_events'] <= before:
                    if recv.done():
                        break
                    await asyncio.sleep(0.1)

                if t['final_events'] <= before:
                    t['error'] = (
                        f'no final transcript for manual turn {i+1}/{count}; '
                        f'messages={t["messages"]}, interim={t["interim_events"]}'
                    )
                    break

                t['turns_finalized'] += 1

            await asyncio.sleep(1)

            if t['turns_sent'] and t['turns_finalized'] == t['turns_sent']:
                t['ok'] = True
            elif not t['error']:
                t['error'] = (
                    f'only {t["turns_finalized"]}/{t["turns_sent"]} manual turns finalized'
                )
    except Exception as e:
        t['error'] = f'{type(e).__name__}: {e}'
    finally:
        if recv is not None and not recv.done():
            recv.cancel()
            try:
                await recv
            except BaseException:
                pass
        t['total_elapsed_sec'] = round(time.monotonic() - started, 3)
    return t


async def run_multiturn_concurrency(client, pcm_path, start_sec, total_sec, concurrency, reference_text):
    started = time.monotonic()
    results = await asyncio.gather(*[
        multi_turn_probe(
            client, pcm_path, start_sec, total_sec,
            f'multi-turn c{concurrency}-job{i+1}'
        )
        for i in range(concurrency)
    ])
    elapsed = time.monotonic() - started
    for r in results:
        r['similarity'] = similarity_metrics(reference_text, r['final_text'])
        r['coverage_ok'] = (
            r['ok']
            and r['similarity']['length_ratio'] >= 0.70
            and r['turns_finalized'] == r['turns_sent']
        )
    return {
        'concurrency': concurrency,
        'success': sum(r['coverage_ok'] for r in results),
        'failed': sum(not r['coverage_ok'] for r in results),
        'elapsed_sec': round(elapsed, 3),
        'results': results,
    }


async def run_concurrency(client, pcm_path, start_sec, seconds, manual_vad, concurrency):
    started = time.monotonic()
    results = await asyncio.gather(*[
        sdk_probe(
            client, pcm_path, start_sec, seconds, [],
            manual_vad, f'c{concurrency}-job{i+1}'
        )
        for i in range(concurrency)
    ])
    elapsed = time.monotonic() - started
    ok = sum(r['ok'] for r in results)
    errors = Counter(r['error'] for r in results if not r['ok'])
    lat = [r['first_message_ms'] for r in results if r['first_message_ms'] is not None]
    return {
        'concurrency': concurrency, 'success': ok, 'failed': concurrency-ok,
        'elapsed_sec': round(elapsed, 3),
        'effective_realtime_x': round(concurrency * seconds / elapsed, 3),
        'median_first_message_ms': sorted(lat)[len(lat)//2] if lat else None,
        'top_error': errors.most_common(1)[0][0] if errors else '',
        'results': results,
    }


def print_concurrency(row):
    err = row['top_error']
    if len(err) > 120:
        err = err[:117] + '...'
    print(
        f"  {row['concurrency']:>11} | {row['success']:>2}/{row['concurrency']:<2} | "
        f"{row['failed']:>6} | {row['elapsed_sec']:>6.1f}s | "
        f"{row['effective_realtime_x']:>7.2f}x | {str(row['median_first_message_ms']):>8} ms | {err}"
    )


async def main():
    report = {
        'started_at_utc': datetime.now(timezone.utc).isoformat(),
        'model': MODEL_ID, 'sdk_target_version': SDK_VERSION,
        'diagnostics': [], 'concurrency': [],
    }
    print(f'{MODEL_NAME} diagnostic + benchmark')
    print('Official control -> your audio -> raw WebSocket isolation -> concurrency.')
    api_key = ask_key()
    src = upload_one_file()
    install_sdk()
    report['google_genai_version'] = google_genai_version
    report['python_version'] = sys.version.split()[0]

    pcm = TMP / f'{int(time.time())}_source.pcm'
    official_wav = TMP / 'google_official_tell_a_story.wav'
    official_pcm = TMP / 'google_official_tell_a_story.pcm'
    client = None
    try:
        print('\n[0] Local audio sanity')
        duration = await asyncio.to_thread(ffmpeg_to_pcm, src, pcm)
        report['source_name'] = src.name
        report['source_duration_sec'] = duration
        print(f'  decoded PCM: mono 16-bit {SAMPLE_RATE}Hz | duration={duration/60:.2f} min')
        probes = candidate_probes(pcm, duration, PROBE_SEC, 12)
        top = probes[:3]
        for i, (offset, stats) in enumerate(top, 1):
            print(
                f"  candidate {i}: offset={offset:.1f}s | RMS={stats['rms']:.0f} | "
                f"dBFS={stats['dbfs']:.1f} | peak={stats['peak']} | zeros={stats['zero_fraction']:.1%}"
            )
        display(Audio(filename=str(write_probe_wav(pcm, top[0][0], PROBE_SEC, TMP/'selected_probe.wav'))))

        client = genai.Client(api_key=api_key)

        print('\n[1] Official Google cookbook control')
        control = None
        raw_control = None
        try:
            await asyncio.to_thread(urllib.request.urlretrieve, OFFICIAL_SAMPLE_URL, official_wav)
            official_duration = await asyncio.to_thread(ffmpeg_to_pcm, official_wav, official_pcm)
            control_sec = min(PROBE_SEC, official_duration)
            control = await sdk_probe(client, official_pcm, 0, control_sec, ['en-US'], False, 'official tell-a-story.wav')
            report['diagnostics'].append(control)
            print_probe(control)
            if not control['ok'] and control['messages'] == 0:
                print('  immediate reconnect check...')
                retry = await sdk_probe(client, official_pcm, 0, control_sec, ['en-US'], False, 'official reconnect retry')
                report['diagnostics'].append(retry)
                print_probe(retry)
                if retry['ok']:
                    control = retry
                    report['control_recovered_after_reconnect'] = True
            if not control['ok']:
                print('  SDK control failed; testing documented raw WebSocket wire format...')
                raw_control = await raw_ws_probe(api_key, official_pcm, 0, control_sec, ['en-US'], 'official raw WebSocket')
                report['diagnostics'].append(raw_control)
                print_probe(raw_control)
        except Exception as e:
            report['official_control_error'] = f'{type(e).__name__}: {e}'
            print('  official control unavailable:', report['official_control_error'])

        print('\n[2] Your recording - correctness matrix')
        source = []
        best_start = top[0][0]
        auto = await sdk_probe(client, pcm, best_start, PROBE_SEC, [], False, 'source best-energy / hybrid VAD')
        source.append(auto); report['diagnostics'].append(auto); print_probe(auto)
        if not auto['ok']:
            manual = await sdk_probe(client, pcm, best_start, PROBE_SEC, [], True, 'source best-energy / manual VAD')
            source.append(manual); report['diagnostics'].append(manual); print_probe(manual)
        if not any(r['ok'] for r in source):
            for i, (offset, _stats) in enumerate(top[1:], 2):
                trial = await sdk_probe(client, pcm, offset, PROBE_SEC, [], False, f'source candidate {i} / hybrid VAD')
                source.append(trial); report['diagnostics'].append(trial); print_probe(trial)
                if trial['ok']:
                    best_start = offset
                    break
        if not any(r['ok'] for r in source) and control and control['ok']:
            raw_source = await raw_ws_probe(api_key, pcm, best_start, PROBE_SEC, [], 'source raw WebSocket')
            source.append(raw_source); report['diagnostics'].append(raw_source); print_probe(raw_source)

        sdk_ok = [r for r in source if r['ok'] and r['transport'] == 'google-genai']
        if not sdk_ok:
            report['classification'] = (
                'SDK_PATH_FAILURE_RAW_WEBSOCKET_OK'
                if raw_control and raw_control['ok']
                else 'SERVICE_OR_SOURCE_PATH_NOT_YET_HEALTHY'
            )
            print('\nSTOP: source did not pass the single-session SDK gate. Concurrency is intentionally skipped.')
            return report

        selected = sdk_ok[0]
        manual_vad = selected['vad_mode'] == 'manual'
        report['classification'] = 'SOURCE_PATH_OK'
        print(f"\n  selected SDK path: VAD={selected['vad_mode']}")

        print('\n[3] Capacity isolation benchmark')
        print('  Every parallel session receives the SAME already-proven speech clip.')
        print('  This isolates concurrency/capacity from source-content differences.')
        passed = [1]
        print('  concurrency | success | failed | elapsed | eff speed | first msg | top error')
        for c in PROFILES:
            print(f'\n  cooldown {COOLDOWN_SEC}s before c={c}...', flush=True)
            await asyncio.sleep(COOLDOWN_SEC)
            row = await run_concurrency(
                client, pcm, best_start, CONCURRENCY_PROBE_SEC, manual_vad, c
            )
            report['concurrency'].append(row)
            print_concurrency(row)
            if row['success'] == c:
                passed.append(c)
            else:
                print('  stopping at first non-100% capacity profile')
                break

        print('\n[4] Transcript completeness reference')
        print(
            f'  Building a {INTEGRITY_SEC}s reference as independent '
            f'{TURN_SEC}s manual turns.'
        )
        reference = await build_independent_reference(
            client, pcm, best_start, INTEGRITY_SEC
        )
        report['integrity_reference'] = reference
        if not reference['ok']:
            report['validated_concurrency'] = None
            print('\nSTOP: could not build a clean independent reference.')
            return report

        print(
            f"  reference: chars={len(normalize_for_compare(reference['text']))} "
            f"segments={len(reference['results'])}"
        )

        print('\n[5] One-session multi-turn integrity')
        multi = await multi_turn_probe(
            client, pcm, best_start, INTEGRITY_SEC,
            'single session / repeated manual turns'
        )
        multi['similarity'] = similarity_metrics(reference['text'], multi['final_text'])
        multi['coverage_ok'] = (
            multi['ok']
            and multi['turns_finalized'] == multi['turns_sent']
            and multi['similarity']['length_ratio'] >= 0.70
        )
        report['multi_turn_integrity'] = multi
        print_probe(multi)
        print(
            f"    turns={multi.get('turns_finalized', 0)}/{multi.get('turns_sent', 0)} "
            f"length_ratio={multi['similarity']['length_ratio']} "
            f"sequence_ratio={multi['similarity']['sequence_ratio']}"
        )

        if not multi['coverage_ok']:
            report['validated_concurrency'] = None
            print(
                '\nSTOP: repeated manual turns in one Live session did not preserve '
                'enough transcript coverage. Production must not use long single turns.'
            )
            return report

        candidate = max(passed)
        print(f'\n[6] Multi-turn sustained concurrency validation: c={candidate}')
        print(
            f'  {candidate} sessions x {INTEGRITY_SEC}s, each internally split into '
            f'{TURN_SEC}s manual turns.'
        )
        await asyncio.sleep(COOLDOWN_SEC)
        mtc = await run_multiturn_concurrency(
            client, pcm, best_start, INTEGRITY_SEC, candidate, reference['text']
        )
        report['multi_turn_concurrency'] = mtc
        print(
            f"  success={mtc['success']}/{candidate} | failed={mtc['failed']} | "
            f"elapsed={mtc['elapsed_sec']:.1f}s"
        )
        for r in mtc['results']:
            print(
                f"    {r['label']}: {'PASS' if r['coverage_ok'] else 'FAIL'} | "
                f"turns={r['turns_finalized']}/{r['turns_sent']} | "
                f"length_ratio={r['similarity']['length_ratio']} | "
                f"seq={r['similarity']['sequence_ratio']} | "
                f"error={r['error'][:120]}"
            )

        if mtc['success'] == candidate:
            floor, count, chunk_len = chunk_floor_minutes(duration, candidate)
            report['validated_concurrency'] = candidate
            report['production_estimate'] = {
                'ideal_floor_min': round(floor, 3),
                'chunk_count': count,
                'chunk_len_sec': round(chunk_len, 3),
                'manual_turn_sec': TURN_SEC,
            }
            print(f'\nRESULT: completeness + concurrency validated at c={candidate}')
            print(
                f'  production target: {count} outer chunks x ~{chunk_len/60:.2f} min, '
                f'each streamed as {TURN_SEC}s manual turns; '
                f'ideal floor ~{floor:.2f} min + turn-finalization overhead.'
            )
        else:
            report['validated_concurrency'] = None
            print(
                '\nRESULT: c=6 capacity was clean on short probes, but completeness '
                'under multi-turn sustained load did not fully validate.'
            )
        return report

        print('\n[5] Sustained concurrency validation with backoff')
        validated = None
        for candidate in reversed(passed):
            print(f'  testing c={candidate} x {long_sec:.0f}s on the same proven long speech clip...')
            await asyncio.sleep(COOLDOWN_SEC)
            sustained = await run_concurrency(
                client, pcm, best_start, long_sec, manual_vad, candidate
            )
            print_concurrency(sustained)
            report.setdefault('sustained_attempts', []).append(sustained)
            if sustained['success'] == candidate:
                validated = candidate
                report['sustained'] = sustained
                break
            print(f'  c={candidate} failed sustained validation; backing off.')

        if validated is not None:
            floor, count, chunk_len = chunk_floor_minutes(duration, validated)
            report['validated_concurrency'] = validated
            report['production_estimate'] = {
                'ideal_floor_min': round(floor, 3),
                'chunk_count': count,
                'chunk_len_sec': round(chunk_len, 3),
            }
            print(f'\nRESULT: validated concurrency={validated}')
            print(
                f'  plan={count} chunks x ~{chunk_len/60:.2f} min; '
                f'ideal floor ~{floor:.2f} min + finalization/retry overhead'
            )
        else:
            report['validated_concurrency'] = None
            print('\nRESULT: no sustained concurrency level passed.')
        return report
    finally:
        if client is not None:
            try:
                client.close()
            except Exception:
                pass
        for p in (pcm, official_pcm):
            try:
                p.unlink(missing_ok=True)
            except Exception:
                pass


report = await main()
report_path = Path('/content/gemini35_live_benchmark_report.json')
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'\nSaved report: {report_path}')
files.download(str(report_path))
